# Chorus -- opinion-CSV echo chamber experiment (Colab)

Runs **only** the opinion-CSV pipeline (`sandbox/opinion_*.py`,
`scripts/run_opinion_experiment.py`) -- agents write free-text opinions to
`initial_stance.csv`, self-report `BELIEF`/`CONFIDENCE` each round, sample
neighbours filtered by belief similarity, and `opinion_round_metrics.csv`
tracks Polarization Index, Global Disagreement, NCI and mean belief shift
per round. This notebook does **not** run the RQ1 bank/generated campaign
(`run_simulation.py`) -- that is a separate pipeline.

Inference is self-hosted via vLLM on the Colab GPU, so this costs GPU time
and nothing else.

**Two hazards**, same ones the RQ1 notebook documents:

1. `asyncio.run()` raises inside a notebook cell, so every actual run is
   launched as a subprocess (`!python scripts/run_opinion_experiment.py ...`),
   never called as a plain Python function from a cell.
2. Colab's filesystem is ephemeral -- mount Drive and point `OUTPUT_DIR`
   at it (Cell 2) if you want `initial_stance.csv` / `opinion_round_metrics.csv`
   to survive the runtime disconnecting.

## 1. Mount Drive (optional, but recommended for persistent output)

In [ ]:
MOUNT_DRIVE = True  # set False to keep everything on the ephemeral local disk

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/chorus_opinion_runs"
else:
    DRIVE_ROOT = "/content/chorus_opinion_runs"

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("output will be written under:", DRIVE_ROOT)

## 2. Get the repo

Clones via HTTPS. If the repo is private, upload a zip instead and unzip it to `/content/Chorus` -- either way this notebook only needs `%cd` to land inside the checkout before Cell 3.

In [ ]:
REPO_URL = "https://github.com/ResearchDrafts/Chorus--Agentic-Network-Orchestration-and-Analysis.git"
REPO_DIR = "/content/Chorus"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

## 3. Install dependencies

In [ ]:
!pip install -q -e ".[dev]"
!pip install -q vllm

## 4. Serve the model with vLLM

Launched as a background subprocess (not `!vllm serve ...` directly, which
would block the cell forever) -- logs go to `/content/vllm_server.log`.
Swap `MODEL` for any model `litellm`'s `hosted_vllm/` provider can reach.

In [ ]:
import subprocess, sys

MODEL = "Qwen/Qwen2.5-7B-Instruct"
PORT = 8000

vllm_log = open("/content/vllm_server.log", "w")
vllm_proc = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--dtype", "half", "--port", str(PORT)],
    stdout=vllm_log, stderr=subprocess.STDOUT,
)
print("vLLM server starting, pid:", vllm_proc.pid)
print("tail /content/vllm_server.log if the health check below times out.")

In [ ]:
import time, urllib.request, urllib.error

API_BASE = f"http://localhost:{PORT}/v1"
MODEL_BACKEND_ID = f"hosted_vllm/{MODEL}"

def wait_for_server(timeout_s=600, poll_s=5):
    deadline = time.time() + timeout_s
    url = f"{API_BASE}/models"
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                body = resp.read().decode("utf-8", "replace")
            if MODEL in body:
                print("server is up and serving", MODEL)
                return
        except urllib.error.URLError:
            pass
        print("waiting for vLLM server...")
        time.sleep(poll_s)
    raise TimeoutError(
        f"vLLM server did not come up within {timeout_s}s -- "
        "check /content/vllm_server.log"
    )

wait_for_server()

## 5. Configure the run

Edit the variables below directly. `LANGUAGES` follows plan.md's rule:
exactly one `True` forces every agent to that language; all four `True`
mixes the population by each persona's own `language preference` in
`data/personas/pool_100.jsonl`; anything else (2 or 3 `True`) is
deliberately unsupported and `run_opinion_experiment.py` will refuse to
start.

In [ ]:
import yaml, os

RUN_ID = "opinion_colab_run1"
TOPIC = "Should city bus fares stay subsidised?"

M = 100     # null (None) uses every persona in the pool
K = 5       # number of update rounds

LANGUAGES = {
    "english": True,
    "hindi": False,
    "hinglish": False,
    "romanized_hindi": False,
}

SIMILARITY_THRESHOLD = 2.0   # belief/v scale, [-2, 2] -- not the x=[-1,1] scale
N = 6                        # neighbours sampled per round, must be <= graph_degree
GRAPH_DEGREE = 6             # even, and < M
GRAPH_REWIRE_P = 0.1
SEED = 1000

OUTPUT_DIR = f"{DRIVE_ROOT}/{RUN_ID}"

config = {
    "run_id": RUN_ID,
    "topic": TOPIC,
    "M": M,
    "K": K,
    "languages": LANGUAGES,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "N": N,
    "graph_degree": GRAPH_DEGREE,
    "graph_rewire_p": GRAPH_REWIRE_P,
    "seed": SEED,
    "model_backend_id": MODEL_BACKEND_ID,
    "api_base": API_BASE,
    "persona_pool_path": "data/personas/pool_100.jsonl",
    "max_reply_tokens": 512,
    "max_concurrency": 32,
    "output_dir": OUTPUT_DIR,
}

CONFIG_PATH = f"/content/{RUN_ID}.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(open(CONFIG_PATH).read())

## 6. Dry run

Validates the config, resolves the language toggle, and confirms the
persona pool parses -- without dispatching a single call. Catches a
partial language-toggle combination, an `M` larger than the pool, or a
malformed persona line before anything is billed.

In [ ]:
!python scripts/run_opinion_experiment.py {CONFIG_PATH} --dry-run

## 7. Run it

Launched as a subprocess (`!python ...`), per Hazard 1 above -- `asyncio.run()`
inside `scripts/run_opinion_experiment.py` cannot be called directly from a
notebook cell. No resume/checkpoint support: if this cell is interrupted,
re-running it starts the whole run over (give it a new `RUN_ID` first if you
don't want to overwrite the partial output).

In [ ]:
!python scripts/run_opinion_experiment.py {CONFIG_PATH}

## 8. Load the results

In [ ]:
import pandas as pd

initial_stance = pd.read_csv(f"{OUTPUT_DIR}/initial_stance.csv")
round_metrics = pd.read_csv(f"{OUTPUT_DIR}/opinion_round_metrics.csv")

print(initial_stance.shape, "agents x columns")
initial_stance.head()

In [ ]:
round_metrics

## 9. Plot Pz / DG / NCI / mean belief shift over rounds

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].plot(round_metrics["round"], round_metrics["polarization_index"], marker="o")
axes[0, 0].set_title("Polarization Index (Pz)")

axes[0, 1].plot(round_metrics["round"], round_metrics["global_disagreement"], marker="o")
axes[0, 1].set_title("Global Disagreement (DG)")

axes[1, 0].plot(round_metrics["round"], round_metrics["nci"], marker="o")
axes[1, 0].set_title("Neighbor Correlation Index (NCI)")

axes[1, 1].plot(round_metrics["round"], round_metrics["mean_belief_shift"], marker="o", label="mean (signed)")
axes[1, 1].plot(round_metrics["round"], round_metrics["mean_abs_individual_shift"], marker="o", label="mean |shift|")
axes[1, 1].set_title("Mean Belief Shift")
axes[1, 1].legend()

for ax in axes.flat:
    ax.set_xlabel("round")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Comparing language conditions

Rerun Cell 5 onward with a different `RUN_ID` and a different `LANGUAGES`
toggle (same `SEED`, same everything else) to get a comparable arm --
`M`/`K`/`seed`/graph settings held identical is what makes two runs'
`opinion_round_metrics.csv` comparable to each other. Each run is a full
end-to-end pass (no resume), so budget GPU time accordingly: roughly
`M * (K + 1)` LLM calls per run.

Remember to `vllm_proc.terminate()` when you are done serving, if you plan
to reuse the GPU for something else in the same session.